# Pipeline de prétraitement

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
import joblib


PROJECT_DIR = Path().cwd().parent.resolve()
DATA_DIR = PROJECT_DIR / "data"
DATA_PATH = DATA_DIR / "03_DONNEES.csv"
OUTPUT_DIR = PROJECT_DIR / "output"
MODELS_DIR = OUTPUT_DIR / "models"
NAIVE_PREPROCESSOR_PATH = MODELS_DIR / "naive_preprocessor.pkl"
FEATURE_ENGINEERING_PREPROCESSOR_PATH = (
    MODELS_DIR / "feature_engineering_preprocessor.pkl"
)

## Chargement des données

In [2]:
df = pd.read_csv(DATA_PATH.as_posix())

X = df.drop(["customerID", "Churn"], axis=1)
y = df["Churn"].copy()

## Création dses pipelines de prétraitement

In [3]:
cat_features = X.select_dtypes(include=["object", "str"]).columns.to_list()
num_features = X.select_dtypes(include=np.number).columns.to_list()

print("Features catégorielles : ", cat_features)
print("Features numériques : ", num_features)

Features catégorielles :  ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract']
Features numériques :  ['SeniorCitizen', 'tenure', 'InternetCharges', 'MonthlyCharges', 'TotalCharges']


In [4]:
cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

naive_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_pipeline, cat_features),
        ("num", num_pipeline, num_features),
    ]
)

In [5]:
cat_features_selected = [
    "gender",
    "Partner",
    # "Dependents",  # Distribution du Churn par modalité très proche de celle de dataset (0.4 points)
    "PhoneService",
    "MultipleLines",
    # "InternetService",  # Redondant avec InternetCharges et les services internets
    # "OnlineSecurity",  # Distribution du Churn par modalité très proche de celle de dataset (0.2 points)
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    # "StreamingTV",  # Distribution du Churn par modalité très proche de celle de dataset (0.4 points)
    "StreamingMovies",
    "Contract",
]
num_features_selected = [
    # "SeniorCitizen",  # Distribution du Churn par modalité très proche de celle de dataset (0.1 points)
    "tenure",
    # "MonthlyCharges",  # Redondant avec InternetCharges
    "InternetCharges",
    # "TotalCharges",  # Transformation x -> (1+x)^{0.33}
]
power_feature = ["TotalCharges"]

print("Features catégorielles sélectionnées : ", cat_features_selected)
print("Features numériques transformées : ", power_feature)
print("Features numériques sélectionnées : ", num_features_selected)

alpha = 0.33


def power_transform(x):
    return np.power(1 + x, alpha)


def inverse_power_transform(y):
    return np.power(y, 1 / alpha) - 1


power_pipeline = Pipeline(
    [
        (
            "power",
            FunctionTransformer(
                func=power_transform,
                inverse_func=inverse_power_transform,
                feature_names_out="one-to-one",
                validate=True,
            ),
        ),
        ("scaler", StandardScaler()),
    ]
)

feature_engineering_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_pipeline, cat_features),
        ("power", power_pipeline, power_feature),
        ("num", num_pipeline, num_features),
    ]
)

Features catégorielles sélectionnées :  ['gender', 'Partner', 'PhoneService', 'MultipleLines', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingMovies', 'Contract']
Features numériques transformées :  ['TotalCharges']
Features numériques sélectionnées :  ['tenure', 'InternetCharges']


## Sauvegarde des pipelines de prétraitement

In [6]:
joblib.dump(naive_preprocessor, NAIVE_PREPROCESSOR_PATH.as_posix())
print("Naive preprocessor pipeline path: ", NAIVE_PREPROCESSOR_PATH.as_posix())

joblib.dump(
    feature_engineering_preprocessor, FEATURE_ENGINEERING_PREPROCESSOR_PATH.as_posix()
)
print(
    "Feature engineering preprocessor pipeline path: ",
    FEATURE_ENGINEERING_PREPROCESSOR_PATH.as_posix(),
)

Naive preprocessor pipeline path:  C:/Users/Administrateur/Documents/DESSAUX_Damien_ECF3/output/models/naive_preprocessor.pkl
Feature engineering preprocessor pipeline path:  C:/Users/Administrateur/Documents/DESSAUX_Damien_ECF3/output/models/feature_engineering_preprocessor.pkl
